### Phase 5: Modeling and Tuning (GroupKFold)

Using Cost-Sensitive Learning and GroupKFold to strictly prevent Spatial Leakage (memorizing static glaciers).

In [7]:
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [8]:
X_pca = np.load("../final_datasets/clean_datasets/X_pca.npy")
y = np.load("../final_datasets/clean_datasets/y.npy")
groups = np.load("../final_datasets/clean_datasets/groups.npy", allow_pickle=True)

scale_weight = np.bincount(y)[0] / np.bincount(y)[1]
print(f"Scale weight: {scale_weight:.2f}")

Scale weight: 9.19


In [9]:
from sklearn.model_selection import GroupKFold, cross_val_score, RandomizedSearchCV
import xgboost as xgb

In [10]:
gkf = GroupKFold(n_splits=5)
xgb_model = xgb.XGBClassifier(scale_pos_weight=scale_weight, random_state=42, eval_metric="logloss")

scores = cross_val_score(xgb_model, X_pca, y, groups=groups, cv=gkf, scoring="roc_auc")
print(f"GroupKFold ROC-AUC: {scores.mean():.4f} (+/- {scores.std():.4f})")

GroupKFold ROC-AUC: 0.9829 (+/- 0.0023)


In [11]:
param_dist = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 4, 5],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.8, 0.9]
}

random_search = RandomizedSearchCV(
    xgb_model,
    param_distributions=param_dist,
    n_iter=10,
    scoring="roc_auc",
    cv=gkf,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_pca, y, groups=groups)
print(f"Best AUC: {random_search.best_score_:.4f}")


Best AUC: 0.9870


In [12]:
import joblib
joblib.dump(random_search.best_estimator_, "../pkl/final_xgb_model.pkl")

['../pkl/final_xgb_model.pkl']